# G9 — RoleNet+: who spoke in the context, and B's previous utterance (round 1 of at most 2)

G8b showed three things:
- RoleNet beats B1 by +4.83 UAR [+2.40, +7.13] (5/5 folds) and beats LateFusion by +3.15.
- The specific value of **role assignment** is not established: RoleNet − noRole is about +1 and not significant.
- The diagnostics found that B's *own* previous utterance predicts B's next emotion best. In clip II, the label matches
  50.6% of the time when B spoke there, 45.2% when a third person did, and about 35% when A did.

RoleNet+ therefore grounds the **speech** in roles, not only the faces. Everything else is identical to RoleNet
(same features, hyper-parameters, folds, and seeds).

1. **Speaker-role tags.** Each utterance in clips I/II gets soft weights for "spoken by A / B / someone else". The
   weights are added to its speech token as a mix of learned role embeddings; clip III is tagged "A".
   * *by A*: the ECAPA voice cosine to clip III.
   * *by B*: a **B-pointer**, a logistic regression over clip I–III cues only. The cues are voice vs A, which faces
     are present, the listener's frame share, mouth–audio sync per role, the number of identities, and voice
     continuity between clips I and II.
   * The pointer's **training labels** come from clip IV's voice. This is used at training time only: clip IV is never
     an input.
   * The pointer is cross-fitted. Eval rows are scored by a model fitted on the training episodes. Training rows are
     scored out-of-fold, grouped by episode, so train and eval inputs have the same quality.
2. **B's previous-utterance token.** This is the pointer-weighted average of the clip I/II speech tokens, with a
   learned "absent" token when no context utterance looks like B's. It has an auxiliary head (weight 0.3) that
   predicts the gold emotion of the context clip B spoke in. That target is also training-only (clip IV voice).

**Arms:**

| Arm | What it is |
|---|---|
| `RoleNet` | Reference: G8b's model, re-run |
| `RoleNet+spk` | Adds speaker-role tags (component 1) |
| `RoleNet+` | Adds speaker-role tags and B's previous-utterance token (components 1 and 2) |
| `RoleNet+ -faceRoles` | `RoleNet+` without face role assignment |
| `RoleNet+ ORACLE` | Analysis only: uses the clip-IV-derived "B spoke" labels at evaluation, so it measures the headroom of a perfect pointer. **Never a result.** |

**Decision rule, fixed before running.** Adopt `RoleNet+` over `RoleNet` for the single test run only if all three
hold:
- its seed-ensemble ΔUAR under LA is > 0;
- its 6-class macro-recall Δ (without fear) is > 0;
- the 6-class Δ is positive in ≥ 4 of 5 folds.

Otherwise keep `RoleNet`. At most one more round follows.

5-fold CV over the 45 train+val episodes, with the same folds and seeds as G8b. The test split stays untouched.

In [ ]:
!pip install -q speechbrain

In [ ]:
# ======== CONFIG ========
import os


def first_existing(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"none of {paths}")


DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv",
                           "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv")
G8A_DIR = first_existing("/kaggle/input/datasets/ptrnghieu/g8a-features", "/kaggle/input/g8a-features")
OUT_DIR = "/kaggle/working"

N_OUTER, N_INNER_DEV = 5, 5
SEEDS = [42, 123, 456]
# RoleNet, identical to G8b (set a priori, not tuned)
RN = dict(D=128, heads=4, layers=2, dropout=0.2, lr=3e-4, wd=1e-2, epochs=80, patience=12, batch=64,
          aux_w=0.3, a_w=0.3, p_drop_ctx=0.3, p_drop_face=0.15)
BPREV_W = 0.3                # auxiliary weight of the B-previous-utterance head
PCA_DIM, MAXF, MAXF_POOL = 128, 24, 32
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25
LA_TAU = 1.0
VOICE_SAME_COS = 0.35
DEBUG_PER_EPISODE = None

FULL = dict(role=True, faces=True, ctx=True, aux=True, mdrop=True, spk=False, bprev=False, oracle=False)
EXPERIMENTS = [
    ("RoleNet",             FULL),
    ("RoleNet+spk",         {**FULL, 'spk': True}),
    ("RoleNet+",            {**FULL, 'spk': True, 'bprev': True}),
    ("RoleNet+ -faceRoles", {**FULL, 'spk': True, 'bprev': True, 'role': False}),
    ("RoleNet+ ORACLE",     {**FULL, 'spk': True, 'bprev': True, 'oracle': True}),   # analysis only
]

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import glob, pickle
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF", "annotation.csv"))[0]
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
sp = sp[sp.split.isin(['train', 'val'])].reset_index(drop=True)      # test rows are dropped here
assert 'test' not in set(sp.split)
if DEBUG_PER_EPISODE:
    sp = sp.groupby('source_folder', group_keys=False).head(DEBUG_PER_EPISODE).reset_index(drop=True)
DEV = sp
train_all = ev = DEV          # names used by the shared feature-loading cell
N = len(DEV)
EPS = np.array(sorted(DEV.source_folder.unique()))
print(f"development MCIS {N} | episodes {len(EPS)}")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ---- load every clip used by any MCIS (I-IV) once, keep it on the GPU
all_clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()) & set(
    f[:-3].replace('_', '/', 1) for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')))
CIDX = {c: i for i, c in enumerate(all_clips)}
missing = [c for c in set(train_all[['clip1', 'clip2', 'clip3']].values.ravel()) | set(ev[['clip1', 'clip2', 'clip3']].values.ravel())
           if c not in CIDX]
assert not missing, f"{len(missing)} clips without features, e.g. {missing[:3]}"

bufs = {k: [] for k in ('face', 'fmask', 'ori', 'text', 'audio', 'afound')}
for c in tqdm(all_clips, desc='loading features'):
    d = torch.load(os.path.join(FEATURES_DIR, c.replace('/', '_') + '.pt'), map_location='cpu', weights_only=False)
    face = d['face_features'].float()
    fm = d.get('face_valid_mask')
    bufs['face'].append(face)
    bufs['fmask'].append(torch.ones(face.shape[0], dtype=torch.bool) if fm is None else torch.as_tensor(fm).bool().reshape(-1))
    bufs['ori'].append(d['ori_features'].float())
    bufs['text'].append(d['text_feature'].float().reshape(-1))
    bufs['audio'].append(d.get('audio_feature', torch.zeros(527)).float().reshape(-1))
    bufs['afound'].append(torch.tensor(bool(d.get('audio_found', True))))
FEAT = {k: torch.stack(v).to(DEVICE) for k, v in bufs.items()}
del bufs
print({k: tuple(v.shape) for k, v in FEAT.items()})


def gather(idx):
    """idx: LongTensor of clip indices (any shape) -> dict of feature tensors with that leading shape."""
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, d=512, n_frames=16, layers=2, heads=8, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_frames, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

    def forward(self, x, mask):  # mask: True = valid frame
        mask = mask.clone()
        mask[~mask.any(1), 0] = True
        h = self.enc(x + self.pos[:, :x.size(1)], src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)


class ClipEncoder(nn.Module):
    """Face/original temporal encoders + text/audio tokens -> 1-layer fusion Transformer -> one 512-d vector."""

    def __init__(self, d=512):
        super().__init__()
        self.face, self.ori = TemporalEncoder(d), TemporalEncoder(d)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.modality = nn.Parameter(torch.randn(1, 4, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, b):
        ori_mask = torch.ones(b['ori'].shape[:2], dtype=torch.bool, device=b['ori'].device)
        tokens = torch.stack([self.face(b['face'], b['fmask']), self.ori(b['ori'], ori_mask),
                              self.text(b['text']), self.audio(F.normalize(b['audio'], dim=-1))], 1)
        valid = torch.ones(tokens.shape[:2], dtype=torch.bool, device=tokens.device)
        valid[:, 3] = b['afound']
        h = self.fusion(tokens + self.modality, src_key_padding_mask=~valid)
        m = valid.unsqueeze(-1).float()
        return self.norm((h * m).sum(1) / m.sum(1))


class ClipRecognizer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.drop = nn.Dropout(0.3)
        self.emo, self.pol = nn.Linear(d, 7), nn.Linear(d, 3)

    def forward(self, b):
        h = self.drop(self.enc(b))
        return self.emo(h), self.pol(h)


N_REC = 12   # 7 emotion probs + 3 polarity probs + max prob + entropy


class Forecaster(nn.Module):
    def __init__(self, use_raw=True, use_traj=False, d=512, positions=None):
        super().__init__()
        self.use_raw, self.use_traj = use_raw, use_traj
        self.positions = positions   # clip positions (0=I, 1=II, 2=III); None = the last n clips
        self.enc = ClipEncoder(d) if use_raw else None
        self.traj = nn.Sequential(nn.LayerNorm(N_REC), nn.Linear(N_REC, d), nn.GELU(), nn.Linear(d, d)) if use_traj else None
        self.clip_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.inter = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d // 2, 7))

    def forward(self, clip_idx, rec):  # clip_idx [B,n], rec [B,n,N_REC], n <= 3 clips in temporal order
        B, n = clip_idx.shape
        tok = 0
        if self.use_raw:
            feats = gather(clip_idx)
            flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
            tok = self.enc(flat).reshape(B, n, -1)
        if self.use_traj:
            tok = tok + self.traj(rec)
        pos = self.clip_pos[:, list(self.positions)] if self.positions is not None else self.clip_pos[:, 3 - n:]
        h = self.inter(tok + pos)
        return self.head(h.mean(1))

## From G8a features to role-tagged slots

In [ ]:
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

G8 = {}
for f in sorted(glob.glob(os.path.join(G8A_DIR, '**', 'shard_*.pkl'), recursive=True)):
    G8.update(pickle.load(open(f, 'rb')))
need = sorted(set(DEV[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in need if c not in G8]
assert not miss, f"{len(miss)} clips missing from G8a, e.g. {miss[:3]}"


def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


# per-clip frame features except the PCA part (18 dims), and the raw HSEmotion embeddings
FB, EMB = {}, {}
for c in need:
    fs = G8[c]['faces']
    if not fs:
        FB[c], EMB[c] = np.zeros((0, 18), np.float32), None
        continue
    dur = max(G8[c]['meta'].get('duration') or 0.0, 1e-3)
    fer = np.stack([d['fer'] for d in fs]).astype(np.float32)
    box = np.stack([d['box'] for d in fs])
    FB[c] = np.concatenate([
        softmax(fer[:, :8]), fer[:, 8:10], np.stack([d['pose'] for d in fs]) / 90.0,
        np.stack([(box[:, 0] + box[:, 2]) / 2, (box[:, 1] + box[:, 3]) / 2,
                  np.sqrt(np.clip((box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]), 0, None))], 1),
        np.nan_to_num(np.array([[d['mouth'] * 10] for d in fs], np.float32)),
        np.array([[min(d['t'] / dur, 1.0)] for d in fs], np.float32)], 1).astype(np.float32)
    EMB[c] = np.stack([d['fer_emb'] for d in fs]) if all('fer_emb' in d for d in fs) else None
HAS_EMB = all(EMB[c] is not None for c in need if len(G8[c]['faces']))
FDIM = (PCA_DIM if HAS_EMB else 0) + 18
print(f"HSEmotion embedding available: {HAS_EMB} | frame feature dim {FDIM}")


def pick_frames(idx, cap):
    return idx if len(idx) <= cap else [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]


def sync(c, face_ids):
    # correlation of mouth opening with the audio energy envelope, and mouth variability, for a set of faces
    A = G8[c]['audio']
    fs = [G8[c]['faces'][j] for j in face_ids]
    fs = [d for d in fs if np.isfinite(d['mouth'])]
    if A is None or len(fs) < 4:
        return 0.0, 0.0
    m = np.array([d['mouth'] for d in fs])
    env = A['env']
    e = np.array([env[min(int(d['t'] * 10), len(env) - 1)] for d in fs]) if len(env) else np.zeros(len(fs))
    r = float(np.corrcoef(m, e)[0, 1]) if m.std() > 1e-6 and e.std() > 1e-9 else 0.0
    return r, float(m.std() * 10)


def mean12(c_faces, n_sampled):
    if not c_faces:
        return np.zeros(12, np.float32)
    fer = np.stack([d['fer'] for d in c_faces]).astype(np.float32)
    v = np.concatenate([softmax(fer[:, :8]), fer[:, 8:10]], 1).mean(0)
    return np.concatenate([v, [1.0, len({d['frame'] for d in c_faces}) / max(n_sampled, 1)]]).astype(np.float32)


NVOICE = 3 * 2 + 3
SLOT, PSLOT = {}, {}                      # (n, role, clip) / (n, clip) -> (clip id, face indices)
FMASK = np.zeros((N, 3, 3, MAXF), bool); PMASK = np.zeros((N, 1, 3, MAXF_POOL), bool)
VOI = np.zeros((N, 3, NVOICE), np.float32)
LRF = np.zeros((N, 60), np.float32)       # G6b-style role means (late fusion and the joint-branch arm)
CENT_COS = np.full(N, np.nan, np.float32)
for n, row in enumerate(tqdm(DEV.itertuples(), total=N, desc='roles')):
    cl = [row.clip1, row.clip2, row.clip3]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8[c]['faces']))]
    lab = np.zeros(len(items), int)
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else None
    if len(items) > 1:
        lab = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                      distance_threshold=1 - SAME_PERSON_COS).fit_predict(E)
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    if A is not None and L is not None:
        ca, cb = E[lab == A].mean(0), E[lab == L].mean(0)
        CENT_COS[n] = float(ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb) + 1e-9))
    role = lambda p: 0 if p == A else (1 if p == L else 2)
    by = defaultdict(list)
    for (k, j), p in zip(items, lab):
        by[(role(p), k)].append(j)
        by[('pool', k)].append(j)
    for k, c in enumerate(cl):
        for r in range(3):
            idx = pick_frames(sorted(by[(r, k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF)
            SLOT[(n, r, k)] = (c, idx); FMASK[n, r, k, :len(idx)] = True
        idx = pick_frames(sorted(by[('pool', k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF_POOL)
        PSLOT[(n, k)] = (c, idx); PMASK[n, 0, k, :len(idx)] = True
        v = [x for r in range(3) for x in sync(c, by[(r, k)])]
        a3, ak = G8[cl[2]]['audio'], G8[c]['audio']
        ok = a3 is not None and ak is not None and a3.get('ecapa') is not None and ak.get('ecapa') is not None
        vcos = float(a3['ecapa'].astype(np.float32) @ ak['ecapa'].astype(np.float32)) if ok else 0.0
        VOI[n, k] = v + [vcos, float(ok), len(set(lab)) / 5.0]

    def faces_of(k, p):
        return [G8[cl[k]]['faces'][j] for (kk, j), q in zip(items, lab) if kk == k and q == p]

    def dominant(k):
        ns = G8[cl[k]]['meta']['n_sampled']
        cand = sorted({q for (kk, q) in frames if kk == k}, key=lambda q: -len(frames[(k, q)]))
        return cand[0] if cand and len(frames[(k, cand[0])]) / max(ns, 1) >= DOMINANT_MIN_FRAC else None

    n3 = G8[cl[2]]['meta']['n_sampled']
    blocks = [mean12(faces_of(2, A) if A is not None else [], n3), mean12(faces_of(2, L) if L is not None else [], n3),
              mean12((faces_of(0, L) + faces_of(1, L)) if L is not None else [], n3)]
    for k in (1, 0):
        d = dominant(k)
        blocks.append(mean12(faces_of(k, d) if d is not None else [], G8[cl[k]]['meta']['n_sampled']))
    LRF[n] = np.concatenate(blocks)

VIS = FMASK[:, 1, 2].any(-1)
print(f"A found in III {FMASK[:, 0, 2].any(-1).mean() * 100:.1f}% | listener visible in III {VIS.mean() * 100:.1f}% | "
      f"listener also in I/II {FMASK[:, 1, :2].any((-1, -2)).mean() * 100:.1f}%")


def build_face_tensors(fit_clips):
    # PCA of the HSEmotion embedding fitted on faces of the training clips of the current fold only
    pca, var = None, None
    if HAS_EMB:
        pool = np.concatenate([EMB[c] for c in fit_clips if EMB.get(c) is not None])
        pick = np.random.default_rng(0).choice(len(pool), min(60000, len(pool)), replace=False)
        pca = PCA(PCA_DIM, random_state=0).fit(pool[pick].astype(np.float32))
        var = float(pca.explained_variance_ratio_.sum())
    FV = {}
    for c in need:
        if len(FB[c]) == 0:
            FV[c] = np.zeros((0, FDIM), np.float32)
        else:
            FV[c] = np.concatenate([pca.transform(EMB[c].astype(np.float32)), FB[c]], 1) if HAS_EMB else FB[c]
    Fa = np.zeros((N, 3, 3, MAXF, FDIM), np.float16)
    Pa = np.zeros((N, 1, 3, MAXF_POOL, FDIM), np.float16)
    for (n, r, k), (c, idx) in SLOT.items():
        if idx:
            Fa[n, r, k, :len(idx)] = FV[c][idx]
    for (n, k), (c, idx) in PSLOT.items():
        if idx:
            Pa[n, 0, k, :len(idx)] = FV[c][idx]
    return torch.tensor(Fa, device=DEVICE), torch.tensor(Pa, device=DEVICE), var

## Who spoke in clips I/II: training labels from clip IV voice, inference cues from clips I–III

In [ ]:
import librosa
try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier
spk_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir=f"{OUT_DIR}/ecapa",
                                           run_opts={"device": DEVICE})
AUDIO_ROOTS = [os.path.join(r, 'audio') for r in glob.glob(os.path.join(DATASET_DIR, '*', 'Hi-EF'))]


def audio_path(c):
    ep, num = c.split('/')
    for root in AUDIO_ROOTS:
        for ext in ('.mp3', '.wav', '.flac', '.m4a'):
            p = os.path.join(root, ep, num + ext)
            if os.path.exists(p):
                return p
    return None


V4 = {}          # clip IV voice: used ONLY to build training labels / targets and the ORACLE analysis arm
for c in tqdm(sorted(set(DEV.clip4)), desc='clip IV voice (training labels only)'):
    p = audio_path(c)
    if p is None:
        continue
    try:
        wav, _ = librosa.load(p, sr=16000, mono=True)
    except Exception:
        continue
    if len(wav) < 8000:
        continue
    with torch.no_grad():
        e = spk_model.encode_batch(torch.tensor(wav, dtype=torch.float32).unsqueeze(0)).reshape(-1).cpu().numpy()
    V4[c] = (e / (np.linalg.norm(e) + 1e-9)).astype(np.float32)
del spk_model; torch.cuda.empty_cache()


def gold(c):
    e = ann.at[c, 7] if c in ann.index else None
    return E2I.get(e, -1) if isinstance(e, str) else -1


def ecapa(c):
    a = G8[c]['audio']
    return a['ecapa'].astype(np.float32) if a is not None and a.get('ecapa') is not None else None


NPF = 16
ISB = np.zeros((N, 2), np.float32); ISB_OK = np.zeros((N, 2), bool)
BTGT = np.full(N, -100, np.int64)
PFEAT = np.zeros((N, 2, NPF), np.float32)
for n, row in enumerate(DEV.itertuples()):
    cl = [row.clip1, row.clip2]
    e4 = V4.get(row.clip4)
    for k, c in enumerate(cl):
        ek, eo = ecapa(c), ecapa(cl[1 - k])
        if e4 is not None and ek is not None:
            ISB[n, k] = float(ek @ e4 >= VOICE_SAME_COS); ISB_OK[n, k] = True
        v = VOI[n, k]
        PFEAT[n, k] = [v[6], v[7], FMASK[n, 1, k].sum() / MAXF, FMASK[n, 0, k].sum() / MAXF, FMASK[n, 2, k].sum() / MAXF,
                       v[0], v[2], v[4], v[1], v[3], v[5], v[8], float(k), float(FMASK[n, 1, 2].any()),
                       float(ek @ eo) if ek is not None and eo is not None else 0.0, v[2] - max(v[0], v[4])]
    for k in (1, 0):                      # prefer clip II
        if ISB[n, k] and gold(cl[k]) >= 0:
            BTGT[n] = gold(cl[k]); break
PA = np.where(VOI[:, :2, 7] > 0, 1 / (1 + np.exp(-(VOI[:, :2, 6] - VOICE_SAME_COS) * 20)), 0.0).astype(np.float32)
print(f"B spoke in clip I {ISB[ISB_OK[:, 0], 0].mean() * 100:.1f}% | clip II {ISB[ISB_OK[:, 1], 1].mean() * 100:.1f}% "
      f"(clip IV voice found {len(V4)}/{DEV.clip4.nunique()}) | rows with a B-previous-utterance target {(BTGT >= 0).sum()}")

## Models

In [ ]:
T = lambda a, dt=None: torch.tensor(a, device=DEVICE) if dt is None else torch.tensor(a, dtype=dt, device=DEVICE)
FMASK, PMASK, VOI = T(FMASK), T(PMASK), T(VOI)
FACE = POOL = PBT = None             # set per fold
CLIPIDX = T([[CIDX[c] for c in r] for r in DEV[['clip1', 'clip2', 'clip3']].values])
TXT, AUD, AFD = FEAT['text'][CLIPIDX], F.normalize(FEAT['audio'][CLIPIDX], dim=-1), FEAT['afound'][CLIPIDX].float()
fm = FEAT['fmask'][CLIPIDX].unsqueeze(-1).float()
SCN = torch.cat([FEAT['ori'][CLIPIDX].mean(2), (FEAT['face'][CLIPIDX] * fm).sum(2) / fm.sum(2).clamp(min=1)], -1)
YB, YA, BT = T(DEV.yB.values), T(DEV.yA.values), T(BTGT)
PAT, ISBT = T(PA), T(ISB)


class FramePool(nn.Module):
    def __init__(self, fin, d):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(fin), nn.Linear(fin, d), nn.GELU(), nn.Linear(d, d))
        self.score = nn.Linear(d, 1)

    def forward(self, x, m):
        h = self.proj(x.float())
        a = self.score(h).squeeze(-1).masked_fill(~m, -1e4)
        w = torch.softmax(a, -1) * m.float()
        return (w.unsqueeze(-1) * h).sum(-2), m.any(-1)


class RoleNetPlus(nn.Module):
    # RoleNet (G8b) + optional speaker-role tags on speech tokens and a B-previous-utterance token
    def __init__(self, cfg, d=RN['D']):
        super().__init__()
        self.cfg, self.R = cfg, (3 if cfg['role'] else 1)
        self.pool = FramePool(FDIM, d)
        self.absent = nn.Parameter(torch.randn(self.R, 3, d) * 0.02)
        self.face_role = nn.Parameter(torch.randn(self.R, d) * 0.02)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.voice = nn.Linear(NVOICE, d)
        self.scene = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, d))
        self.ctx_role = nn.Parameter(torch.randn(2, d) * 0.02)
        self.clip_emb = nn.Parameter(torch.randn(3, d) * 0.02)
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        if cfg['spk']:
            self.spk_role = nn.Parameter(torch.randn(3, d) * 0.02)            # spoken by A / B / other
        if cfg['bprev']:
            self.bprev_absent = nn.Parameter(torch.randn(1, d) * 0.02)
            self.bprev_emb = nn.Parameter(torch.randn(1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, RN['heads'], 4 * d, RN['dropout'], batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, RN['layers'], enable_nested_tensor=False)
        mk = lambda: nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 7))
        self.head, self.head_face, self.head_ctx = mk(), mk(), mk()
        self.head_A = mk() if cfg['role'] else None
        self.head_bprev = mk() if cfg['bprev'] else None

    def forward(self, ix, train=False):
        B, aux = len(ix), {}
        x, m = (FACE[ix], FMASK[ix]) if self.R == 3 else (POOL[ix], PMASK[ix])
        h, present = self.pool(x, m)
        h = torch.where(present.unsqueeze(-1), h, self.absent.unsqueeze(0).expand(B, -1, -1, -1))
        h = h + self.face_role[None, :, None] + self.clip_emb[None, None]
        ft = h.reshape(B, self.R * 3, -1)
        aux['face'] = (self.head_face(ft.mean(1)), YB[ix], RN['aux_w'])
        if self.head_A is not None:
            tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
            aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        spk_ = self.text(TXT[ix]) + self.audio(AUD[ix]) * AFD[ix].unsqueeze(-1) + self.voice(VOI[ix]) + self.ctx_role[0]
        ctx = []
        if self.cfg['spk'] or self.cfg['bprev']:
            pA = PAT[ix]
            pB = (ISBT[ix] if self.cfg['oracle'] else PBT[ix]) * (1 - pA)
            pO = (1 - pA - pB).clamp(min=0)
        if self.cfg['spk']:
            w = torch.stack([pA, pB, pO], -1)                                     # [B, 2, 3]
            w3 = torch.cat([w, torch.tensor([1.0, 0, 0], device=w.device).expand(B, 1, 3)], 1)
            spk_ = spk_ + w3 @ self.spk_role
        scn = self.scene(SCN[ix]) + self.ctx_role[1]
        ctx += [spk_ + self.clip_emb, scn + self.clip_emb]
        if self.cfg['bprev']:
            s = pB.sum(1, keepdim=True)
            hb = (pB.unsqueeze(-1) * spk_[:, :2]).sum(1) / s.clamp(min=1e-3)
            hb = torch.where(s > 0.05, hb, self.bprev_absent.expand(B, -1)) + self.bprev_emb
            ctx.append(hb.unsqueeze(1))
            aux['bprev'] = (self.head_bprev(hb), BT[ix], BPREV_W)
        ct = torch.cat(ctx, 1)
        aux['ctx'] = (self.head_ctx(ct.mean(1)), YB[ix], RN['aux_w'])
        toks = torch.cat([self.query.expand(B, -1, -1), ft, ct], 1)
        valid = torch.ones(toks.shape[:2], dtype=torch.bool, device=toks.device)
        if train and self.cfg['mdrop']:
            u = torch.rand(B, device=toks.device)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            nf = ft.shape[1]
            valid[:, 1:1 + nf] &= ~drop_face.unsqueeze(1)
            valid[:, 1 + nf:] &= ~drop_ctx.unsqueeze(1)
        out = self.enc(toks, src_key_padding_mask=~valid)
        return self.head(out[:, 0]), aux


print("parameters:", {n: f"{sum(p.numel() for p in RoleNetPlus(c).parameters()) / 1e6:.2f}M" for n, c in EXPERIMENTS})


def predict(model, ix, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ix), bs):
            out.append(F.softmax(model(ix[i:i + bs])[0], -1).cpu())
    return torch.cat(out).numpy()


def train_eval(cfg, tr, dev, te, seed):
    seed_all(seed)
    model = RoleNetPlus(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=RN['lr'], weight_decay=RN['wd'])
    y_dev = YB[dev].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(RN['epochs']):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), RN['batch']):
            j = perm[i:i + RN['batch']]
            logits, aux = model(j, train=True)
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        u = war_uar(predict(model, dev).argmax(1), y_dev, 7)[1]
        if u > best:
            best, bad = u, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= RN['patience']:
                break
    model.load_state_dict(best_state)
    return predict(model, te), best

## 5-fold episode cross-validation (same folds and seeds as G8b)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

sizes = DEV.source_folder.value_counts()
order = list(sizes.index)
random.Random(0).shuffle(order)
order = sorted(order, key=lambda e: -sizes[e])
load_, FOLD = [0] * N_OUTER, {}
for e in order:
    f = int(np.argmin(load_)); FOLD[e] = f; load_[f] += sizes[e]
fold_of_row = DEV.source_folder.map(FOLD).values
print("fold sizes (MCIS):", load_, "(must match G8b)")

y_all = DEV.yB.values
src = DEV.source_folder.values
X2, Y2, OK2, G2 = PFEAT.reshape(N * 2, -1), ISB.reshape(-1), ISB_OK.reshape(-1), np.repeat(src, 2)


def pointer_probs(tr_rows, te_rows):
    # B-pointer: eval rows scored by a model fitted on training episodes; training rows scored out-of-fold
    rows2 = lambda r: np.sort(np.concatenate([r * 2, r * 2 + 1]))
    out = np.zeros(N * 2, np.float32)
    tr2 = rows2(tr_rows)
    fit2 = tr2[OK2[tr2]]
    sc = StandardScaler().fit(X2[fit2])
    lr_ = lambda idx: LogisticRegression(max_iter=3000, C=1.0).fit(sc.transform(X2[idx]), Y2[idx])
    te2 = rows2(te_rows)
    out[te2] = lr_(fit2).predict_proba(sc.transform(X2[te2]))[:, 1]
    for a, b in GroupKFold(5).split(tr2, groups=G2[tr2]):
        fa = tr2[a][OK2[tr2[a]]]
        out[tr2[b]] = lr_(fa).predict_proba(sc.transform(X2[tr2[b]]))[:, 1]
    return out.reshape(N, 2)


OOF = {name: np.full((len(SEEDS), N, 7), np.nan, np.float32) for name, _ in EXPERIMENTS}
PB_OOF = np.zeros((N, 2), np.float32)
LOGPI = np.zeros((N, 7), np.float32)
log = []
t0 = time.time()
for f in range(N_OUTER):
    tr_eps = [e for e in EPS if FOLD[e] != f]
    dev_eps = sorted(random.Random(100 + f).sample(tr_eps, N_INNER_DEV))
    trr = np.where(np.isin(src, tr_eps))[0]
    fit_rows = np.where(np.isin(src, tr_eps) & ~np.isin(src, dev_eps))[0]
    dev_rows = np.where(np.isin(src, dev_eps))[0]
    te_rows = np.where(fold_of_row == f)[0]
    FACE, POOL, var = build_face_tensors(sorted(set(DEV.iloc[trr][['clip1', 'clip2', 'clip3']].values.ravel())))
    pb = pointer_probs(trr, te_rows)
    PB_OOF[te_rows] = pb[te_rows]
    PBT = T(pb)
    LOGPI[te_rows] = np.log((np.bincount(y_all[trr], minlength=7) + 1) / (len(trr) + 7))
    ok = ISB_OK[te_rows]
    auc = roc_auc_score(ISB[te_rows][ok], pb[te_rows][ok]) if len(set(ISB[te_rows][ok])) == 2 else float('nan')
    print(f"fold {f}: train {len(fit_rows)} | early-stop {len(dev_rows)} | eval {len(te_rows)} | B-pointer AUC {auc:.3f}",
          flush=True)
    tr, dev, te = T(fit_rows), T(dev_rows), T(te_rows)
    for name, cfg in EXPERIMENTS:
        for si, seed in enumerate(SEEDS):
            p, sel = train_eval(cfg, tr, dev, te, seed + 1000 * f)
            OOF[name][si, te_rows] = p
            w, u = war_uar(p.argmax(1), y_all[te_rows], 7)
            log.append({'fold': f, 'exp': name, 'seed': seed, 'sel_UAR': sel, 'UAR': u, 'WAR': w})
            print(f"fold {f} {name:<20} seed {seed}: sel {sel:5.2f} | UAR {u:5.2f} WAR {w:5.2f} | "
                  f"{(time.time() - t0) / 60:.1f} min", flush=True)
            torch.cuda.empty_cache()

assert all(not np.isnan(v).any() for v in OOF.values())
pd.DataFrame(log).to_csv(f"{OUT_DIR}/g9_fold_seed_log.csv", index=False)
np.savez(f"{OUT_DIR}/g9_oof_probs.npz", sample_id=DEV.sample_id.values, fold=fold_of_row, logpi=LOGPI, pb=PB_OOF,
         isb=ISB, isb_ok=ISB_OK, **{k.replace('-', '_').replace('+', 'plus').replace(' ', '_'): v for k, v in OOF.items()})

## Results and the preregistered decision

In [ ]:
FEAR = E2I['fear']


def recalls(p, y):
    return np.array([(p[y == c] == c).mean() * 100 if (y == c).any() else np.nan for c in range(7)])


def uar7(p, y):
    return np.nanmean(recalls(p, y))


def uar6(p, y):
    return np.nanmean(np.delete(recalls(p, y), FEAR))


rng_ = np.random.default_rng(0)
GRP = [np.where(src == e)[0] for e in np.unique(src)]
BOOT = [np.concatenate([GRP[j] for j in rng_.integers(0, len(GRP), len(GRP))]) for _ in range(2000)]


def delta(pa, pb, m, metric):
    idx_m = np.where(m)[0]
    d0 = metric(pa[m], y_all[m]) - metric(pb[m], y_all[m])
    ds = []
    for b in BOOT:
        i = b[m[b]]
        ds.append(metric(pa[i], y_all[i]) - metric(pb[i], y_all[i]))
    lo, hi = np.nanpercentile(ds, [2.5, 97.5])
    return d0, lo, hi


def safe_auc(y, p):
    return roc_auc_score(y, p) if len(set(y)) == 2 else float('nan')


ok = ISB_OK.reshape(-1)
print(f"B-pointer AUC (out-of-fold): all {safe_auc(ISB.reshape(-1)[ok], PB_OOF.reshape(-1)[ok]):.3f} | "
      f"clip I {safe_auc(ISB[ISB_OK[:, 0], 0], PB_OOF[ISB_OK[:, 0], 0]):.3f} | "
      f"clip II {safe_auc(ISB[ISB_OK[:, 1], 1], PB_OOF[ISB_OK[:, 1], 1]):.3f}")

LOGP = {k: np.log(v.mean(0) + 1e-9) for k, v in OOF.items()}
PRED = {'plain': {k: v.argmax(1) for k, v in LOGP.items()}, 'LA': {k: (v - LA_TAU * LOGPI).argmax(1) for k, v in LOGP.items()}}
print(f"\n== per-seed pooled UAR, plain ==")
for k, v in OOF.items():
    per = [uar7(v[s].argmax(1), y_all) for s in range(len(v))]
    print(f"  {k:<20} " + " ".join(f"{u:5.2f}" for u in per) + f"  (mean {np.mean(per):.2f})")
print("\n== seed ensemble: UAR LA | 6-class LA | UAR plain | WAR LA ==")
for k in PRED['LA']:
    print(f"  {k:<20} {uar7(PRED['LA'][k], y_all):6.2f} | {uar6(PRED['LA'][k], y_all):6.2f} | "
          f"{uar7(PRED['plain'][k], y_all):6.2f} | {(PRED['LA'][k] == y_all).mean() * 100:6.2f}")
print("\n== per-class recall, LA ==")
print(f"  {'':<20}" + "".join(f"{e[:7]:>8}" for e in EMO))
for k, p in PRED['LA'].items():
    print(f"  {k:<20}" + "".join(f"{x:8.1f}" for x in recalls(p, y_all)))

CONTRASTS = [("RoleNet+", "RoleNet"), ("RoleNet+spk", "RoleNet"), ("RoleNet+", "RoleNet+spk"),
             ("RoleNet+", "RoleNet+ -faceRoles"), ("RoleNet+ ORACLE", "RoleNet+")]
bspoke = (ISB * ISB_OK).max(1) > 0
SUBSETS = {'all': np.ones(N, bool), 'listener visible in III': VIS.copy() if isinstance(VIS, np.ndarray) else VIS,
           'B spoke in I/II (analysis)': bspoke, 'B did not speak in I/II (analysis)': ~bspoke}
for sname, m in SUBSETS.items():
    m = np.asarray(m, bool)
    print(f"\n== paired contrasts, {sname} (n={m.sum()}) — LA 7-class | LA 6-class | plain 7-class ==")
    if m.sum() < 30:
        print("  too few MCIS, skipped")
        continue
    for a, b in CONTRASTS:
        r = [delta(PRED[mode][a], PRED[mode][b], m, met) for mode, met in (('LA', uar7), ('LA', uar6), ('plain', uar7))]
        print(f"  {a:<20} - {b:<20} " + " | ".join(f"{d:+5.2f} [{lo:+5.2f},{hi:+5.2f}]" for d, lo, hi in r))

print("\n== per-fold 6-class Δ (LA), RoleNet+ − RoleNet ==")
fold_d = []
for f in range(N_OUTER):
    m = fold_of_row == f
    fold_d.append(uar6(PRED['LA']['RoleNet+'][m], y_all[m]) - uar6(PRED['LA']['RoleNet'][m], y_all[m]))
    print(f"  fold {f}: {fold_d[-1]:+.2f}")
d7 = uar7(PRED['LA']['RoleNet+'], y_all) - uar7(PRED['LA']['RoleNet'], y_all)
d6 = uar6(PRED['LA']['RoleNet+'], y_all) - uar6(PRED['LA']['RoleNet'], y_all)
adopt = d7 > 0 and d6 > 0 and sum(x > 0 for x in fold_d) >= 4
print(f"\nDECISION (preregistered): ΔUAR LA {d7:+.2f}, Δ6-class {d6:+.2f}, positive folds {sum(x > 0 for x in fold_d)}/5 -> "
      f"{'ADOPT RoleNet+ for the test run' if adopt else 'KEEP RoleNet'}")